# EMG2MCU quickstart: synthetic recording

This notebook runs a lightweight version of the [EMG2MCU](https://github.com/LENeR-UEL/EMG2MCU) pipeline in your
browser. It generates a deterministic three-class synthetic recording, applies the same preprocessing stages as
the real package (bandpass + notch filtering, onset detection, windowing, time-domain features), trains a
Random Forest on the features and inspects the deployable biquad filter the pipeline would ship to the ESP32.

Everything runs in your browser with [Pyodide](https://pyodide.org). Your data never leaves your machine.
Change the synthetic recording parameters, the filter cutoffs, the window size, the model or the feature set,
then press **Run all**.

In [ ]:
import numpy as np
import pandas as pd

from scipy.signal import butter, filtfilt, iirnotch
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from emg2mcu_sandbox import display, show_events, show_features, show_filter, show_models, show_signal

rng = np.random.default_rng(42)
fs = 1000
channel_names = ["Peito", "Trapézio", "Pescoço"]
label_names = ["Peito", "Trapézio", "Pescoço"]

## Parameters

By default the data are synthetic: three channels (Peito, Trapézio, Pescoço) and three classes (the same names).
Each class is a sum of three sinusoid bursts at slightly different frequencies per channel. Move, add or remove
repetitions to change the size of the recording; the bandpass and notch filters, the onset detector and the
classifier are all configured from the form below.

In [ ]:
n_reps = 4  # @param {type:"slider", min:1, max:8, label:"Repetitions per class"}
segment_seconds = 3.0  # @param {type:"slider", min:1, max:6, step:0.5, label:"Active segment length (s)"}
rest_seconds = 0.5  # @param {type:"slider", min:0, max:2, step:0.1, label:"Standby between segments (s)"}
noise_std = 0.015  # @param {type:"slider", min:0.001, max:0.1, step:0.001, label:"Background noise (std)"}
# @markdown **Filtering**
lowcut = 30  # @param {type:"slider", min:10, max:100, label:"Bandpass lowcut (Hz)"}
highcut = 450  # @param {type:"slider", min:100, max:500, step:10, label:"Bandpass highcut (Hz)"}
notch_freq = 60  # @param [0, 50, 60] {label:"Notch frequency (Hz, 0 = off)"}
# @markdown **Onset detection**
smooth_window_ms = 500  # @param {type:"slider", min:50, max:1000, step:50, label:"Envelope smoothing window (ms)"}
sustain_ms = 1500  # @param {type:"slider", min:200, max:3000, step:100, label:"Sustain before an event is kept (ms)"}
tol_ratio = 0.2  # @param {type:"slider", min:0.05, max:0.5, step:0.05, label:"Envelope threshold (share of range)"}
# @markdown **Windowing and features**
window_ms = 50  # @param {type:"slider", min:10, max:200, step:10, label:"Window length (ms)"}
features = ["RMS", "MAV"]  # @param ["RMS", "MAV", "WL", "ZC"] {label:"Time-domain features per channel"}
# @markdown **Model**
model_name = "RandomForest"  # @param ["LogisticRegression", "KNN (k=5)", "DecisionTree", "RandomForest", "SVM (RBF)"]
folds = 5  # @param {type:"slider", min:2, max:10, label:"Cross-validation folds"}

## Generate the synthetic recording

The same generator as the [EMG2MCU example notebook](https://github.com/LENeR-UEL/EMG2MCU/blob/main/example/EMG2MCU_example.ipynb):
three sinusoid bursts per class, 150 ms fade-in / fade-out, Gaussian background noise, and a standby segment between
repetitions. The output is a `(N, 3)` array of samples (rows × channels).

In [ ]:
patterns = [[0.8, 0.15, 0.15], [0.15, 0.8, 0.15], [0.15, 0.15, 0.8]]
signal_blocks, label_blocks = [], []

def add_segment(label, duration_s, amplitudes):
    n = int(duration_s * fs)
    t = np.arange(n) / fs
    seg = rng.normal(0, noise_std, size=(3, n))
    if label >= 0:
        fade = np.minimum(1, np.minimum(t / 0.15, (duration_s - t) / 0.15))
        freqs = np.array([83, 117, 151]) + 7 * label
        carriers = np.sin(2 * np.pi * freqs[:, None] * t)
        carriers += 0.35 * np.sin(2 * np.pi * freqs[:, None] * 1.7 * t)
        seg += np.asarray(amplitudes)[:, None] * fade * carriers
    signal_blocks.append(seg.T)
    label_blocks.append(np.full(n, label, dtype=int))

add_segment(-1, 2.0, [0, 0, 0])
for _ in range(n_reps):
    for label, amplitudes in enumerate(patterns):
        add_segment(label, segment_seconds, amplitudes)
        add_segment(-1, rest_seconds, [0, 0, 0])

signals = np.concatenate(signal_blocks, axis=0)
labels = np.concatenate(label_blocks)
print(f"recording: {signals.shape[0]} samples × {signals.shape[1]} channels at {fs} Hz")
print(f"classes: {dict(zip(*np.unique(labels, return_counts=True)))}")

## Bandpass + notch filtering

A 4th-order Butterworth bandpass between `lowcut` and `highcut` Hz removes the DC offset and the high-frequency
noise; the optional notch filter at `notch_freq` Hz removes the mains interference (60 Hz in the Americas, 50 Hz in
Europe). Both filters use `filtfilt` here (zero-phase, for benchmark); the on-device filter is causal and uses the
same parameters — see the *Deployable filter* section below.

In [ ]:
nyq = fs / 2
b_bp, a_bp = butter(4, [lowcut / nyq, highcut / nyq], btype="band")
filtered = filtfilt(b_bp, a_bp, signals, axis=0)
if notch_freq:
    b_n, a_n = iirnotch(notch_freq / nyq, 30)
    filtered = filtfilt(b_n, a_n, filtered, axis=0)
else:
    b_n, a_n = np.array([1.0]), np.array([1.0])
show_signal(signals, filtered, fs, channel_names=channel_names)

## Onset detection

A moving-RMS smoother (window `smooth_window_ms` ms) turns the signal into an envelope; an event is an interval
where the envelope stays above `tol_ratio` of its range for at least `sustain_ms` ms.

In [ ]:
env_sq = filtered.mean(axis=1) ** 2
kernel = np.ones(int(smooth_window_ms * fs / 1000)) / max(1, int(smooth_window_ms * fs / 1000))
env = np.sqrt(np.convolve(env_sq, kernel, mode="same"))

thr = env.min() + tol_ratio * (env.max() - env.min())
sustain = int(sustain_ms * fs / 1000)
events, ev_labels = [], []
i, n = 0, len(env)
while i < n:
    if env[i] < thr:
        i += 1
        continue
    j, above = i, 0
    while j < n and env[j] >= thr:
        above += 1; j += 1
    if above >= sustain:
        start = i
        while start > 0 and env[start - 1] >= thr: start -= 1
        if j - start > 0.4 * fs:
            events.append((start, j))
            ev_labels.append(int(labels[(start + j) // 2]))
        i = j
    else:
        i += 1
print(f"detected {len(events)} events")
show_events(env, events, ev_labels, label_names, fs)

## Windowed time-domain features

Each event is split into `window_ms`-ms windows; per window and per channel the chosen features (RMS, MAV, …) are
computed. The feature table is the input of the classifier.

In [ ]:
def feature(w, kind):
    if kind == "RMS": return float(np.sqrt(np.mean(w ** 2)))
    if kind == "MAV": return float(np.mean(np.abs(w)))
    if kind == "WL":  return float(np.mean(np.abs(np.diff(w, axis=0))))
    if kind == "ZC":  return float(np.mean(np.abs(np.diff(np.sign(w), axis=0)) > 0))
    raise ValueError(kind)

win = int(window_ms * fs / 1000)
rows, ys = [], []
for (s, e), lab in zip(events, ev_labels):
    if lab < 0: continue
    ev = filtered[s:e]
    for k in range(0, len(ev) - win + 1, win):
        w = ev[k:k + win]
        rows.append({f"{kind}_{name}": feature(w[:, c], kind)
                     for kind in features for c, name in enumerate(channel_names)})
        ys.append(lab)
X = pd.DataFrame(rows).reset_index(drop=True)
y = np.asarray(ys)
print(f"X: {X.shape}, y: {y.shape}, class counts: {dict(zip(*np.unique(y, return_counts=True)))}")
show_features(X, y, label_names)

## Model selection

Cross-validated accuracy, F1 (weighted) and ROC AUC (one-vs-rest) for every algorithm in the configuration. The
package's real pipeline also trains gradient-boosted models (LightGBM, XGBoost, CatBoost); they aren't available
in Pyodide so this notebook compares simpler classifiers instead.

In [ ]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=2000, random_state=0),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "DecisionTree": DecisionTreeClassifier(random_state=0),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=0),
    "SVM (RBF)": SVC(probability=True, random_state=0),
}
rows = []
skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=0)
Xs = StandardScaler().fit_transform(X.values)
for name, model in models.items():
    acc, f1s, aucs = [], [], []
    for tr, te in skf.split(Xs, y):
        model.fit(Xs[tr], y[tr])
        pred = model.predict(Xs[te])
        proba = model.predict_proba(Xs[te]) if hasattr(model, "predict_proba") else None
        acc.append(accuracy_score(y[te], pred))
        f1s.append(f1_score(y[te], pred, average="weighted"))
        if proba is not None and len(np.unique(y[te])) > 1:
            aucs.append(roc_auc_score(y[te], proba, multi_class="ovr", average="weighted"))
        else:
            aucs.append(np.nan)
    rows.append({"model": name, "accuracy": float(np.mean(acc)), "f1": float(np.mean(f1s)), "roc_auc": float(np.nanmean(aucs))})
results = pd.DataFrame(rows)
show_models(results)

## Deployable filter (ESP32)

The same Butterworth bandpass and notch used offline are expressed as second-order sections (biquads) for the
on-device filter. The package generates the biquad coefficients and a skeleton Arduino sketch that runs the
filter on the chip; the figures below show the coefficients this notebook just used and the equivalent C++ code.

In [ ]:
show_filter(b_bp, a_bp, b_n if notch_freq else np.array([1.0]), a_n if notch_freq else np.array([1.0]))
from emg2mcu_sandbox import show_code
arduino = f"""
// Generated by the EMG2MCU sandbox on synthetic data; the real package fills these
// coefficients from the chosen PreprocessingConfig (lowcut={lowcut} Hz, highcut={highcut} Hz,
// notch={notch_freq} Hz, fs={fs} Hz).
const float b_bandpass[] = {{{', '.join(f'{c:.6f}f' for c in b_bp)}}};
const float a_bandpass[] = {{{', '.join(f'{c:.6f}f' for c in a_bp)}}};
const float b_notch[]    = {{{', '.join(f'{c:.6f}f' for c in (b_n if notch_freq else [1.0]))}}};
const float a_notch[]    = {{{', '.join(f'{c:.6f}f' for c in (a_n if notch_freq else [1.0]))}}};

float biquad_step(const float* b, const float* a, float x, float* z) {{
    float y = b[0]*x + z[0];
    z[0] = b[1]*x - a[1]*y + z[1];
    z[1] = b[2]*x - a[2]*y;
    return y;
}}

void loop() {{
    int raw = analogRead({34});
    static float z_bp[2] = {{0, 0}}, z_n[2] = {{0, 0}};
    float v = biquad_step(b_bandpass, a_bandpass, raw, z_bp);
    if ({notch_freq}) v = biquad_step(b_notch, a_notch, v, z_n);
    // feed `v` to your event detector / feature extractor
}}
""".strip()
show_code(arduino, language="cpp", caption="ESP32 biquad filter skeleton (bandpass + notch, fs=1 kHz)")

## Your turn

Add a cell below to explore further. The model ranking is in `results`; the feature matrix in `X`; the detected
events in `events`. Plotly figures and matplotlib plots are shown inline. To use the full package locally,
install it with `pip install emg2mcu` and replace the in-memory pipeline with `import EMG2MCU`.